In [ ]:
!pip install -q langchain langchain-groq

import os

from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")

Groq connected successfully!


In [ ]:
SYSTEM = """
ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Plum cake costs Rs 500.
Chocolate cake costs Rs 650.
One cupcake costs Rs 80.
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise a delivery time or place.
If delivery information is not available,
say "I'll check with the owner."
Calculate totals carefully.
Never invent menu items or prices.

FORMAT:
Answer in under 50 words.
Be friendly and polite.
Reply in English or Telugu according to the customer.
"""

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []


def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer


print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))

print(ask("What's my name?"))

Yes, all our cakes—including plum, chocolate, and cupcakes—are available in eggless versions. Feel free to let us know which one you’d like!
I don’t have that information. If you’d like to share it, I’ll be happy to address you by name!


In [ ]:
print(ask("What is the price of a plum cake?"))

print(ask("What is the price of 6 cupcakes?"))

print(ask("Can I get a cake today evening?"))

print(ask("Do you deliver to Kakinada?"))

A plum cake is Rs 500.
Six cupcakes cost Rs 480.
I’m sorry, but we need 24‑hour notice for orders. I’ll check with the owner and let you know if a cake can be ready for this evening.
I’m not sure about delivery to Kakinada. I’ll check with the owner and get back to you.


In [ ]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480
How much is a chocolate cake?,650
What is the price of one cupcake?,80
Can you deliver in 30 minutes?,check with the owner
Do you have eggless chocolate cake?,eggless
How much are 2 plum cakes?,1000

Writing tests.csv


In [ ]:
import csv

def run_tests(test_chain, path="tests.csv"):

    with open(path, "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = test_chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"]

        ok = expected.lower() in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:100]
        )

    total = len(rows)

    print("\nResults:", passed, "/", total)
    print("Pass rate:", f"{passed / total:.0%}")


run_tests(chain)

PASS | How much is a plum cake? -> A plum cake costs Rs 500. Let me know if you’d like to place an order!
PASS | Do you make eggless cakes? -> Yes, all our cakes—including plum, chocolate, and cupcakes—are available in eggless versions. Feel f
FAIL | Can I get a cake today evening? -> I’m sorry, we need 24‑hour notice for orders. I’ll check with the owner and get back to you.
PASS | Do you deliver to Kakinada? -> I’ll check with the owner and get back to you. Thank you for reaching out!
PASS | What is the price of 6 cupcakes? -> The total for 6 cupcakes is Rs 480. Let me know if you’d like to place an order!
PASS | How much is a chocolate cake? -> Chocolate cake costs Rs 650. Let me know if you’d like to place an order!
PASS | What is the price of one cupcake? -> One cupcake costs Rs 80.
PASS | Can you deliver in 30 minutes? -> I’ll check with the owner.
FAIL | Do you have eggless chocolate cake? -> Yes, we offer an egg‑less chocolate cake for Rs 650. Please note we need a 24‑hour noti

In [ ]:
SYSTEM_V2 = """
ROLE:
You are the helpful customer assistant for Sweet Crumbs Home Bakery.

FACTS:
Plum cake costs Rs 500.
Chocolate cake costs Rs 650.
One cupcake costs Rs 80.
All cakes have eggless versions.
Every order requires at least 24 hours' notice.

RULES:
For same-day orders, clearly mention the 24-hour notice.
Never promise delivery to any location.
For delivery questions, reply:
"I'll check with the owner."
Never invent prices or availability.
Calculate totals correctly.

FORMAT:
Use friendly and simple English or Telugu.
Keep answers under 50 words.
"""

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_V2),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain_v2 = prompt_v2 | llm | StrOutputParser()

run_tests(chain_v2)

PASS | How much is a plum cake? -> A plum cake costs Rs 500.
PASS | Do you make eggless cakes? -> Yes! All our cakes – plum, chocolate, cupcakes, etc. – are available in eggless versions.
FAIL | Can I get a cake today evening? -> I’m sorry, but we need at least 24 hours’ notice for orders. Please place your order earlier so we c
PASS | Do you deliver to Kakinada? -> I’ll check with the owner.
PASS | What is the price of 6 cupcakes? -> Rs 480.
PASS | How much is a chocolate cake? -> Chocolate cake costs Rs 650.
PASS | What is the price of one cupcake? -> One cupcake costs Rs 80.
PASS | Can you deliver in 30 minutes? -> I'll check with the owner.
FAIL | Do you have eggless chocolate cake? -> Yes! We have an egg‑less chocolate cake. It costs Rs 650. Please give us at least 24 hours’ notice f
PASS | How much are 2 plum cakes? -> Rs 1000 for two plum cakes.

Results: 8 / 10
Pass rate: 80%


PROMPT IMPROVEMENT EXPERIMENT

Before improvement:
Pass rate = 70%

After improvement:
Pass rate = 80%

Observation:
I improved the system prompt by making the
delivery rules and 24-hour notice clearer.

I compared both test results to understand
whether the new prompt improved the responses.

In [ ]:
llm_temp = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain_temp = prompt_v2 | llm_temp | StrOutputParser()

print("FIRST RUN")

run_tests(chain_temp)

print("\nSECOND RUN")

run_tests(chain_temp)

FIRST RUN
PASS | How much is a plum cake? -> ప్లమ్ కేక్ ధర రూ. 500.
FAIL | Do you make eggless cakes? -> Yes, every cake we offer—including the plum, chocolate, and cupcakes—has an egg‑less version.
FAIL | Can I get a cake today evening? -> I’m sorry, but we need at least 24‑hour notice for any cake order. For today’s evening, we can’t tak
PASS | Do you deliver to Kakinada? -> I'll check with the owner.
PASS | What is the price of 6 cupcakes? -> Six cupcakes cost Rs 480.
PASS | How much is a chocolate cake? -> Chocolate cake costs Rs 650.
PASS | What is the price of one cupcake? -> A single cupcake costs **Rs 80**.
PASS | Can you deliver in 30 minutes? -> I'll check with the owner.
FAIL | Do you have eggless chocolate cake? -> Yes, we have an egg‑less chocolate cake. It costs Rs 650. Let us know your order details – we need a
PASS | How much are 2 plum cakes? -> ₹1000 for two plum cakes.

Results: 7 / 10
Pass rate: 70%

SECOND RUN
PASS | How much is a plum cake? -> A plum cake costs Rs

TEMPERATURE EXPERIMENT

I changed the temperature from 0.3 to 1.0.

I ran the tests twice and compared the answers.

Observation:
The model may give different wording and responses
when the temperature is increased.

Higher temperature generally allows more variation
in the generated responses.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM_V2
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, all our cakes—including plum, chocolate, and cupcakes—are available in egg‑less versions.


BUSINESS JOBS AND AI APPROACH

1. Calculate the total price of cakes.
   Approach: Plain Python.

2. Write a festival promotional message.
   Approach: One LLM call.

3. Handle a complicated customer order that
   requires multiple checks and actions.
   Approach: AI Agent.

In [ ]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}


def get_price(item: str, qty: int = 1) -> int:
    """
    Calculate the total price of a bakery item.

    Args:
        item: Name of the bakery item.
        qty: Number of items.

    Returns:
        Total price in rupees.
    """

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not available")

    if qty <= 0:
        raise ValueError("Quantity must be positive")

    return MENU[item] * qty


def can_accept_order(hours_notice: int) -> bool:
    """
    Check whether an order has 24 hours' notice.

    Args:
        hours_notice: Number of hours before the order.

    Returns:
        True if 24 hours' notice is available.
    """

    return hours_notice >= 24


print(get_price("plum cake", 2))

print(can_accept_order(48))

print(can_accept_order(5))

1000
True
False
